# Optimización de procesos con Python para ingenieros

Cuaderno del curso. Cada celda lleva el mismo número que en el video (por ejemplo, **Celda 2.4**).

Para ejecutar una celda: clic en ella y Mayúsculas + Enter. Las celdas se corren en orden: algunas usan lo que definió una celda anterior.

## Sección 0 · Bienvenida y configuración

**Celda 0.1** · Carga las librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
print("NumPy", np.__version__)
print("SciPy", scipy.__version__)

**Celda 0.2** · Una función de Python

In [ ]:
def costo(x):
    return (x - 3)**2 + 2
print(costo(1))
print(costo(3))

**Celda 0.3** · Un arreglo: muchos valores a la vez

In [ ]:
x = np.linspace(0, 6, 7)
print(x)
print(costo(x))

**Celda 0.4** · Tu primera gráfica

In [ ]:
x = np.linspace(0, 6, 121)
plt.plot(x, costo(x))
plt.xlabel("x")
plt.ylabel("costo")
plt.show()

**Celda 0.5** · Tu primer optimizador

In [ ]:
from scipy.optimize import minimize_scalar
res = minimize_scalar(costo)
print("x óptimo:", round(res.x, 4))
print("costo mínimo:", round(res.fun, 4))

## Sección 1 · Cuántas decisiones tienes: grados de libertad

**Celda 1.1** · Las variables de una corriente

In [ ]:
NC = 3   # componentes: metanol, DME y agua
variables_corriente = NC + 2   # flujo, NC - 1 composiciones, T y P
print("variables de una corriente:", variables_corriente)

**Celda 1.2** · El mezclador

In [ ]:
def gl_mezclador(NC):
    incognitas = NC + 2        # la corriente de salida
    ecuaciones = NC + 1 + 1    # balances, energía y presión
    return incognitas - ecuaciones
print("mezclador con NC = 3:", gl_mezclador(3))

**Celda 1.3** · El divisor

In [ ]:
def gl_divisor(NC):
    incognitas = 2 * (NC + 2)            # las dos salidas
    ecuaciones = 2*(NC - 1) + 4 + 1      # composiciones, T y P, balance total
    return incognitas - ecuaciones
print("divisor con NC = 3:", gl_divisor(3))

**Celda 1.4** · El flash

In [ ]:
def gl_flash(NC):
    incognitas = 2*NC + 3     # V, y, L, x, T, P y Q
    ecuaciones = 2*NC + 1     # balances, equilibrio y energía
    return incognitas - ecuaciones
print("flash con NC = 3:", gl_flash(3))

**Celda 1.5** · Tres unidades y tres valores de NC

In [ ]:
print("unidad      NC=2  NC=3  NC=10")
for nombre, gl in [("mezclador", gl_mezclador),
                   ("divisor  ", gl_divisor),
                   ("flash    ", gl_flash)]:
    print(nombre, "  ", gl(2), "   ", gl(3), "   ", gl(10))

**Celda 1.6** · El tren precalentador + flash, desde cero

In [ ]:
NC = 3
incognitas = (NC + 2) + 1 + (2*NC + 3)   # corriente 2, calor Q_H y flash
ecuaciones = (NC + 2) + (2*NC + 1)       # precalentador y flash
print("incógnitas:", incognitas)
print("ecuaciones:", ecuaciones)
print("GL del tren:", incognitas - ecuaciones)

**Celda 1.7** · El mismo tren, sumando unidades

In [ ]:
gl_precalentador = 1
gl_flash_tren = gl_flash(3)   # la corriente compartida se cuenta una vez
print("GL del tren:", gl_precalentador + gl_flash_tren)

## Sección 2 · Del enunciado al modelo de optimización

**Celda 2.1** · El equilibrio de la reacción del DME

In [ ]:
R = 8.314                       # J/(mol K)
dH, dG = -23400.0, -16800.0     # J/mol, a 25 °C
lnK298 = -dG / (R * 298.15)
def K(T):                       # constante de equilibrio
    return np.exp(lnK298 + dH/R * (1/298.15 - 1/T))
def X_eq(T):                    # conversión de equilibrio
    return 2*np.sqrt(K(T)) / (1 + 2*np.sqrt(K(T)))
print("X_eq a 300 °C:", round(X_eq(573.15), 4))

**Celda 2.2** · El costo contra la temperatura del reactor

In [ ]:
def costo_dme(Tc):                      # $/t de DME
    T = Tc + 273.15
    reactor = 6.0 * np.exp(80000/R * (1/T - 1/523.15))
    reciclo = 45.0 * (1 - X_eq(T)) / X_eq(T)
    calor = 0.02 * (Tc - 200)
    return reactor + reciclo + calor
print("costo a 250 °C:", round(costo_dme(250), 2))

**Celda 2.3** · Un barrido cada 25 grados

In [ ]:
temperaturas = np.arange(200, 401, 25)
costos = costo_dme(temperaturas)
for t, c in zip(temperaturas, costos):
    print(t, "°C:", round(c, 2))
print("la más barata del barrido:", temperaturas[np.argmin(costos)], "°C")

**Celda 2.4** · Grafica la curva completa antes de optimizar

In [ ]:
T_fina = np.linspace(200, 400, 401)
plt.plot(T_fina, costo_dme(T_fina))
plt.plot(temperaturas, costos, "o")
plt.xlabel("temperatura del reactor (°C)")
plt.ylabel("costo ($/t de DME)")
plt.show()

**Celda 2.5** · El costo del aislante

In [ ]:
def costo_aislante(e):          # e: espesor en metros
    energia = 2.592 / e          # la pérdida de calor que se paga
    capital = 250 * e            # el aislante, anualizado
    return energia + capital
print(round(costo_aislante(0.05), 2))
print(round(costo_aislante(0.20), 2))

**Celda 2.6** · Grafica antes de optimizar

In [ ]:
espesores = np.linspace(0.02, 0.30, 281)
plt.plot(espesores, costo_aislante(espesores), label="total")
plt.plot(espesores, 2.592 / espesores, "--", label="energía")
plt.plot(espesores, 250 * espesores, "--", label="capital")
plt.xlabel("espesor (m)")
plt.legend()
plt.show()

**Celda 2.7** · El mínimo del barrido

In [ ]:
costos_e = costo_aislante(espesores)
i = np.argmin(costos_e)
print("espesor:", round(espesores[i], 3), "m")
print("costo:", round(costos_e[i], 2))

**Celda 2.8** · El primer óptimo con minimize_scalar

In [ ]:
from scipy.optimize import minimize_scalar
res = minimize_scalar(costo_aislante, bounds=(0.005, 0.5), method="bounded")
print("e* =", round(res.x * 100, 2), "cm")
print("C* =", round(res.fun, 2))
print("éxito:", res.success, "· evaluaciones:", res.nfev)

**Celda 2.9** · Compara con la solución a mano

In [ ]:
e_mano = np.sqrt(2.592 / 250)     # donde la derivada vale cero
print("a mano:", round(e_mano * 100, 2), "cm")
print("diferencia:", round(abs(e_mano - res.x) * 1000, 4), "mm")

## Sección 3 · Optimizar una variable

**Celda 3.1** · La razón dorada

In [ ]:
tau = (np.sqrt(5) - 1) / 2
print("tau =", round(tau, 5))
print("tau al cuadrado =", round(tau**2, 5))
print("uno menos tau =", round(1 - tau, 5))

**Celda 3.2** · Cuatro iteraciones a mano sobre el DME

In [ ]:
a, b = 200.0, 400.0
for k in range(4):
    x1, x2 = b - tau*(b - a), a + tau*(b - a)
    print(k, "intervalo", round(a, 1), round(b, 1), "· puntos", round(x1, 2), round(x2, 2))
    if costo_dme(x1) < costo_dme(x2): b = x2   # el mínimo no está a la derecha de x2
    else: a = x1                                # el mínimo no está a la izquierda de x1

**Celda 3.3** · La sección dorada con un punto reciclado

In [ ]:
def dorada(f, a, b, tol=1.0):
    x1, x2 = b - tau*(b - a), a + tau*(b - a)
    f1, f2, n = f(x1), f(x2), 2
    while b - a > tol:
        if f1 < f2: b, x2, f2 = x2, x1, f1; x1 = b - tau*(b - a); f1 = f(x1)
        else:       a, x1, f1 = x1, x2, f2; x2 = a + tau*(b - a); f2 = f(x2)
        n += 1
    return (a + b) / 2, n

**Celda 3.4** · La temperatura óptima con la sección dorada

In [ ]:
T_dorada, n_dorada = dorada(costo_dme, 200, 400)
print("T* =", round(T_dorada, 2), "°C")
print("costo:", round(costo_dme(T_dorada), 2), "$/t")
print("evaluaciones:", n_dorada)

**Celda 3.5** · El horno: costo y sus derivadas

In [ ]:
def C_horno(t):        # mil $/año; t: meses entre limpiezas
    return 2160/t + 95*(np.exp(0.22*t) - 1)
def C1(t):             # primera derivada
    return -2160/t**2 + 20.9*np.exp(0.22*t)
def C2(t):             # segunda derivada
    return 4320/t**3 + 4.598*np.exp(0.22*t)
print("costo con limpieza cada 4 meses:", round(C_horno(4), 2))

**Celda 3.6** · Newton, iteración por iteración

In [ ]:
t = 4.0
for k in range(4):
    paso = C1(t) / C2(t)
    t = t - paso
    print(k, "t =", round(t, 4), "· paso =", round(-paso, 4))
print("t* =", round(t, 4), "meses · costo =", round(C_horno(t), 2))

**Celda 3.7** · ¿Y si limpiamos cada seis meses?

In [ ]:
extra = C_horno(6) - C_horno(t)
print("costo extra:", round(extra, 2), "mil $/año")
print("en porcentaje:", round(100 * extra / C_horno(t), 2), "%")

**Celda 3.8** · Cuándo se pierde Newton

In [ ]:
def d1(x): return 3*x**2 - 12*x + 9     # derivada de x³ − 6x² + 9x + 2
def d2(x): return 6*x - 12              # segunda derivada
for x0 in (2.5, 1.8):
    x = x0
    for k in range(5): x = x - d1(x) / d2(x)
    print("desde", x0, "termina en", round(x, 4), "· segunda derivada:", round(d2(x), 2))

**Celda 3.9** · Brent con minimize_scalar

In [ ]:
res_b = minimize_scalar(costo_dme, bracket=(200, 300, 400),
                        method="brent", options={"xtol": 1e-4})
print("T* =", round(res_b.x, 3), "°C · costo =", round(res_b.fun, 2), "$/t")
print("éxito:", res_b.success, "· evaluaciones:", res_b.nfev)

**Celda 3.10** · Tres resultados lado a lado

In [ ]:
print("barrido cada 25 °C:", temperaturas[np.argmin(costos)], "°C ·", len(temperaturas), "evaluaciones")
print("sección dorada:", round(T_dorada, 2), "°C ·", n_dorada, "evaluaciones")
print("Brent:", round(res_b.x, 3), "°C ·", res_b.nfev, "evaluaciones")

**Celda 3.11** · Práctica: el aerorrefrigerante con tarifa por bloques

In [ ]:
def C_aire(x):                     # mil $/año; x: aire en m³/s
    energia = 8 * x                # MWh/año del ventilador
    tarifa = 1.6*min(energia, 400) + 2.8*max(energia - 400, 0)
    return 40000/x + tarifa
res_a = minimize_scalar(C_aire, bounds=(20, 80), method="bounded")
print("x* =", round(res_a.x, 2), "m³/s · costo =", round(res_a.fun, 2), "mil $/año")

## Sección 4 · Optimizar varias variables

**Celda 4.1** · El sobrecosto fuera del punto de diseño

In [ ]:
def sobrecosto(v):          # mil $/año
    dT, dpurga = v          # desviación de temperatura (°C) y de purga (p.p.)
    return dT**2 + 6*dpurga**2
def gradiente(v):
    return np.array([2*v[0], 12*v[1]])
x0 = np.array([6.0, 1.0])
print("sobrecosto:", sobrecosto(x0), "· gradiente:", gradiente(x0))

**Celda 4.2** · El mapa de curvas de nivel

In [ ]:
X, Y = np.meshgrid(np.linspace(-7, 7, 201), np.linspace(-2, 2, 201))
plt.contour(X, Y, X**2 + 6*Y**2, levels=[1, 5, 10, 20, 42])
plt.plot(6, 1, "o")
plt.xlabel("desviación de temperatura (°C)")
plt.ylabel("desviación de purga (p.p.)")
plt.show()

**Celda 4.3** · Descenso con paso fijo

In [ ]:
x = x0.copy()
for k in range(6):
    x = x - 0.15 * gradiente(x)
    print(k + 1, np.round(x, 3), "· sobrecosto:", round(sobrecosto(x), 3))

**Celda 4.4** · Armijo: cuánto avanzar

In [ ]:
def armijo(x, d, alfa=1.0, c1=1e-4, rho=0.5):
    pendiente = gradiente(x) @ d
    while sobrecosto(x + alfa*d) > sobrecosto(x) + c1*alfa*pendiente:
        alfa = rho * alfa        # paso rechazado: lo partimos a la mitad
    return alfa

**Celda 4.5** · Descenso con Armijo

In [ ]:
x = x0.copy()
for k in range(4):
    d = -gradiente(x)
    alfa = armijo(x, d)
    x = x + alfa * d
    print(k + 1, "alfa =", alfa, np.round(x, 4), "· sobrecosto:", round(sobrecosto(x), 4))

**Celda 4.6** · Newton en varias variables

In [ ]:
H = np.diag([2.0, 12.0])            # la matriz hessiana
x_newton = x0 - np.linalg.solve(H, gradiente(x0))
print("después de un paso:", x_newton, "· sobrecosto:", sobrecosto(x_newton))

**Celda 4.7** · La derivada por diferencias finitas

In [ ]:
e0, exacta = 0.05, -2.592/0.05**2 + 250
print("derivada exacta:", round(exacta, 4))
for h in [1e-2, 1e-3, 1e-4, 1e-6]:
    central = (costo_aislante(e0 + h) - costo_aislante(e0 - h)) / (2*h)
    print("h =", h, "· error:", f"{central - exacta:.1e}")

**Celda 4.8** · Con el ruido de un simulador

In [ ]:
rng = np.random.default_rng(7)
def con_ruido(e):          # error de una parte en un millón, como un simulador
    return costo_aislante(e) * (1 + 1e-6*rng.uniform(-1, 1))
for h in [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]:
    errores = [(con_ruido(e0+h) - con_ruido(e0-h))/(2*h) - exacta for _ in range(2000)]
    print("h =", h, "· error típico:", f"{np.sqrt(np.mean(np.square(errores))):.1e}")

**Celda 4.9** · Tres reactores: la potencia total

In [ ]:
def potencia(v):             # kW; flujos a los reactores 1 y 2, en kmol/h
    x1, x2 = v
    x3 = 100 - x1 - x2        # el tercero recibe lo que falta para 100
    return 0.02*x1**2 + 0.03*x2**2 + 0.025*x3**2
print("potencia con 20, 50 y 30 kmol/h:", potencia([20, 50]), "kW")

**Celda 4.10** · minimize con BFGS: el resultado completo

In [ ]:
from scipy.optimize import minimize
res_bfgs = minimize(potencia, [20, 50], method="BFGS")
print(res_bfgs)

**Celda 4.11** · minimize con Nelder-Mead

In [ ]:
res_nm = minimize(potencia, [20, 50], method="Nelder-Mead")
print("x =", np.round(res_nm.x, 2), "· potencia =", round(res_nm.fun, 2), "kW")
print("éxito:", res_nm.success, "· evaluaciones:", res_nm.nfev)
print("BFGS usó", res_bfgs.nfev, "evaluaciones")
x1, x2 = res_bfgs.x; x3 = 100 - x1 - x2
print("potencias marginales:", round(0.04*x1, 2), round(0.06*x2, 2), round(0.05*x3, 2))

**Celda 4.12** · Variables con escalas muy distintas

In [ ]:
def sobrecosto_P(v):        # mil $/año; v = presión (Pa) y fracción de purga
    P, z = v
    return ((P - 2.0e5)/1.0e4)**2 + 6*((z - 0.03)/0.01)**2
res_mal = minimize(sobrecosto_P, [1.5e5, 0.05], method="BFGS")
print("éxito:", res_mal.success, "·", res_mal.message)
print("evaluaciones:", res_mal.nfev)
print("P =", round(res_mal.x[0]), "Pa · purga =", round(res_mal.x[1], 4))

**Celda 4.13** · Escala cada variable con su rango típico

In [ ]:
escala = np.array([1.0e4, 0.01])
u0 = np.array([1.5e5, 0.05]) / escala
res_bien = minimize(lambda u: sobrecosto_P(u * escala), u0, method="BFGS")
print("éxito:", res_bien.success, "· evaluaciones:", res_bien.nfev)
print("P =", round(res_bien.x[0] * escala[0]), "Pa · purga =", round(res_bien.x[1] * escala[1], 4))

## Sección 5 · Programación lineal

**Celda 5.1** · La planta de solventes en linprog

In [ ]:
from scipy.optimize import linprog
c = [-40, -30]           # $/lote; linprog minimiza, así que usamos −Z
A_ub = [[1, 1],          # acabado: horas por lote
        [2, 1]]          # reactor: horas por lote
b_ub = [40, 60]          # horas disponibles por semana
res_p = linprog(c, A_ub=A_ub, b_ub=b_ub, method="highs")
print("premium:", res_p.x[0], "· estándar:", res_p.x[1], "lotes por semana")
print("utilidad:", -res_p.fun, "$ por semana")

**Celda 5.2** · Los cuatro vértices, uno por uno

In [ ]:
for premium, estandar in [(0, 0), (30, 0), (20, 20), (0, 40)]:
    print((premium, estandar), "→ utilidad:", 40*premium + 30*estandar, "$")

**Celda 5.3** · El estado y las holguras

In [ ]:
print("estado:", res_p.status, "·", res_p.message)
print("horas que sobran (acabado, reactor):", res_p.ineqlin.residual)

**Celda 5.4** · La mezcla de gasolinas

In [ ]:
c_mezcla = [14, 10, 8]               # $/bbl de A, B y C
A_oct = [[-95, -88, -83]]            # octanaje ≥ 90, multiplicado por −1
b_oct = [-9000]
res_m = linprog(c_mezcla, A_ub=A_oct, b_ub=b_oct, A_eq=[[1, 1, 1]], b_eq=[100], method="highs")
print("A, B, C:", res_m.x.round(2), "bbl")
print("costo:", round(res_m.fun, 2), "$")

**Celda 5.5** · Los precios sombra de la planta

In [ ]:
print("acabado y reactor:", -res_p.ineqlin.marginals, "$ por hora extra")

**Celda 5.6** · Compruébalo con una hora más

In [ ]:
for horas in ([41, 60], [40, 61]):
    r = linprog(c, A_ub=A_ub, b_ub=horas, method="highs")
    print(horas, "horas → utilidad:", -r.fun, "$")

**Celda 5.7** · El precio sombra del octanaje

In [ ]:
print("precio sombra:", round(-res_m.ineqlin.marginals[0], 3), "$ por octano·bbl")
r89 = linprog(c_mezcla, A_ub=A_oct, b_ub=[-8900], A_eq=[[1, 1, 1]], b_eq=[100], method="highs")
print("con 89 octanos:", round(r89.fun, 2), "$ · ahorro:", round(res_m.fun - r89.fun, 2), "$")

**Celda 5.8** · Hasta dónde vale un precio sombra

In [ ]:
for horas in (29, 30, 59, 60):
    r = linprog(c, A_ub=A_ub, b_ub=[horas, 60], method="highs")
    print(horas, "h de acabado → precios sombra:", -r.ineqlin.marginals)

**Celda 5.9** · Sin solución: solo hay 20 bbl de A

In [ ]:
cotas = [(0, 20), (0, None), (0, None)]   # sin cotas, linprog supone de 0 a infinito
r = linprog(c_mezcla, A_ub=A_oct, b_ub=b_oct, A_eq=[[1, 1, 1]], b_eq=[100],
            bounds=cotas, method="highs")
print("estado:", r.status, "·", r.message)

**Celda 5.10** · Sin límite: olvidamos las restricciones de capacidad

In [ ]:
r = linprog(c, method="highs")
print("estado:", r.status, "·", r.message)

## Sección 6 · Programación no lineal con restricciones

**Celda 6.1** · El tanque por sustitución

In [ ]:
def area(r):                       # m², con la altura ya sustituida
    return 2*np.pi*r**2 + 200/r
res_t = minimize_scalar(area, bounds=(0.5, 10), method="bounded")
r_t = res_t.x; h_t = 100 / (np.pi * r_t**2)
print("r* =", round(r_t, 3), "m · h* =", round(h_t, 3), "m · área =", round(res_t.fun, 2), "m²")
print("altura entre diámetro:", round(h_t / (2*r_t), 3))

**Celda 6.2** · El tanque con su restricción, en minimize

In [ ]:
def area2(v): return 2*np.pi*v[0]**2 + 2*np.pi*v[0]*v[1]
volumen = {"type": "eq", "fun": lambda v: np.pi*v[0]**2*v[1] - 100}
res_L = minimize(area2, [2, 5], method="SLSQP", constraints=[volumen])
print("r, h =", res_L.x.round(3), "m · área =", round(res_L.fun, 2), "m²")
print("multiplicador:", res_L.multipliers.round(3))

**Celda 6.3** · Comprueba que los gradientes son paralelos

In [ ]:
r, h = res_L.x
grad_area = np.array([4*np.pi*r + 2*np.pi*h, 2*np.pi*r])
grad_vol = np.array([2*np.pi*r*h, np.pi*r**2])
print("cocientes:", (grad_area / grad_vol).round(4))
print("lambda a mano, 2 entre r:", round(2 / r, 4))

**Celda 6.4** · Un metro cúbico más

In [ ]:
volumen101 = {"type": "eq", "fun": lambda v: np.pi*v[0]**2*v[1] - 101}
res101 = minimize(area2, [2, 5], method="SLSQP", constraints=[volumen101])
print("área con 101 m³:", round(res101.fun, 2), "m²")
print("diferencia:", round(res101.fun - res_L.fun, 3), "m²")

**Celda 6.5** · KKT: un techo de cuatro metros

In [ ]:
techo = {"type": "ineq", "fun": lambda v: 4 - v[1]}     # SciPy pide g(x) ≥ 0
res_K = minimize(area2, [3, 3.5], method="SLSQP", constraints=[volumen, techo])
print("r, h =", res_K.x.round(3), "m · área =", round(res_K.fun, 2), "m²")
print("multiplicadores (volumen, techo):", res_K.multipliers.round(3))

**Celda 6.6** · Con un techo de seis metros

In [ ]:
techo6 = {"type": "ineq", "fun": lambda v: 6 - v[1]}
res_K6 = minimize(area2, [3, 3.5], method="SLSQP", constraints=[volumen, techo6])
print("r, h =", res_K6.x.round(3), "m · área =", round(res_K6.fun, 2), "m²")
print("multiplicadores (volumen, techo):", res_K6.multipliers.round(3))

**Celda 6.7** · Los reactores gemelos

In [ ]:
def agitacion(x): return 0.02*x[0]**2 + 0.03*x[1]**2      # kW
carga = {"type": "eq", "fun": lambda x: x[0] + x[1] - 100}  # kmol/h en total
res_g = minimize(agitacion, [50, 50], method="SLSQP", constraints=[carga])
print("cargas:", res_g.x.round(2), "kmol/h · potencia:", round(res_g.fun, 2), "kW")
print("multiplicador:", res_g.multipliers.round(3))
print("marginales:", round(0.04*res_g.x[0], 2), round(0.06*res_g.x[1], 2))

**Celda 6.8** · Un tope de 55 kmol/h en el primer reactor

In [ ]:
tope = {"type": "ineq", "fun": lambda x: 55 - x[0]}
res_g55 = minimize(agitacion, [50, 50], method="SLSQP", constraints=[carga, tope])
print("cargas:", res_g55.x.round(2), "kmol/h · potencia:", round(res_g55.fun, 2), "kW")
print("multiplicadores (carga, tope):", res_g55.multipliers.round(3))
print("marginales:", round(0.04*res_g55.x[0], 2), round(0.06*res_g55.x[1], 2))

**Celda 6.9** · Economía de escala: el arranque importa

In [ ]:
def capital(x): return 10*x[0]**0.6 + 10*x[1]**0.6    # miles de $/año, regla de los seis décimos
for x0 in ([50, 50], [55, 45], [30, 70]):
    r = minimize(capital, x0, method="SLSQP", constraints=[carga], bounds=[(10, 90), (10, 90)])
    print("desde", x0, "→", r.x.round(1), "· costo:", round(r.fun, 2), "· éxito:", r.success)

**Celda 6.10** · Un QP es un sistema lineal

In [ ]:
M = np.array([[0.04, 0, -1],       # 0.04 x1 − lambda = 0
              [0, 0.06, -1],       # 0.06 x2 − lambda = 0
              [1, 1, 0]])          # x1 + x2 = 100
b = np.array([0, 0, 100])
x1, x2, lam = np.linalg.solve(M, b)
print("x1 =", round(x1, 2), "· x2 =", round(x2, 2), "· lambda =", round(lam, 3))

## Sección 7 · Ajustar un modelo a datos

**Celda 7.1** · Datos de presión de vapor del metanol

In [ ]:
A_nist, B_nist, C_nist = 8.07919, 1581.341, 239.650   # NIST (Ambrose y Sprake, 1970): log10 P[mmHg], T en °C
T_dat = np.linspace(15, 84, 12)                       # °C, dentro del rango de las constantes
rng = np.random.default_rng(7)                        # semilla fija: siempre los mismos datos
ruido = 1 + 0.005*rng.standard_normal(12)             # medio por ciento de error de medición
P_dat = 10**(A_nist - B_nist/(T_dat + C_nist)) * ruido   # mmHg
print(np.column_stack([T_dat.round(1), P_dat.round(1)]))

**Celda 7.2** · La suma de cuadrados depende de los parámetros

In [ ]:
def antoine(T, A, B, C): return A - B/(T + C)          # log10 de la presión en mmHg
def suma_cuadrados(p):
    return np.sum((np.log10(P_dat) - antoine(T_dat, *p))**2)
print("con las constantes de NIST:", f"{suma_cuadrados([A_nist, B_nist, C_nist]):.3e}")
print("con C = 220:", f"{suma_cuadrados([A_nist, B_nist, 220]):.3e}")

**Celda 7.3** · Ajustar es minimizar: la suma con minimize

In [ ]:
res_sc = minimize(suma_cuadrados, [8, 1600, 240], method="Nelder-Mead",
                  options={"maxiter": 4000, "xatol": 1e-8, "fatol": 1e-14})
print("A, B, C:", res_sc.x.round(3))
print("suma:", f"{res_sc.fun:.3e}", "· evaluaciones:", res_sc.nfev, "· éxito:", res_sc.success)

**Celda 7.4** · curve_fit paso a paso

In [ ]:
from scipy.optimize import curve_fit
p_aj, pcov = curve_fit(antoine, T_dat, np.log10(P_dat), p0=[8, 1600, 240])
print("A, B, C ajustados:", p_aj.round(3))
print("A, B, C de NIST:  ", [A_nist, B_nist, C_nist])
print("suma de cuadrados:", f"{suma_cuadrados(p_aj):.3e}")

**Celda 7.5** · El ajuste y sus residuos

In [ ]:
residuos = np.log10(P_dat) - antoine(T_dat, *p_aj)
fig, (arriba, abajo) = plt.subplots(2, 1, sharex=True)
arriba.plot(T_dat, P_dat, "o"); arriba.plot(T_dat, 10**antoine(T_dat, *p_aj)); arriba.set_ylabel("P (mmHg)")
abajo.axhline(0, color="gray"); abajo.plot(T_dat, residuos, "o"); abajo.set_ylabel("residuo")
abajo.set_xlabel("T (°C)"); plt.show()
print("signos de los residuos:", np.sign(residuos).astype(int))

**Celda 7.6** · Un modelo que no alcanza: sin la C

In [ ]:
def clausius(T, A, B): return A - B/(T + 273.15)        # Clausius–Clapeyron, sin la C
p_cc, _ = curve_fit(clausius, T_dat, np.log10(P_dat), p0=[8, 2000])
res_cc = np.log10(P_dat) - clausius(T_dat, *p_cc)
r2_cc = 1 - np.sum(res_cc**2) / np.sum((np.log10(P_dat) - np.log10(P_dat).mean())**2)
print("R² sin la C:", round(r2_cc, 5), "· signos de los residuos:", np.sign(res_cc).astype(int))
plt.axhline(0, color="gray"); plt.plot(T_dat, res_cc, "o")
plt.xlabel("T (°C)"); plt.ylabel("residuo"); plt.show()

**Celda 7.7** · Datos de equilibrio de acetona y cloroformo

In [ ]:
# Gao y otros (2018), Applied Sciences 8, 1519, tabla A1 · 101.3 kPa · 1 = acetona, 2 = cloroformo
x1 = np.array([0.102, 0.202, 0.301, 0.396, 0.494, 0.587, 0.679, 0.786, 0.893])
y1 = np.array([0.067, 0.160, 0.280, 0.401, 0.544, 0.660, 0.765, 0.866, 0.945])
T_K = np.array([335.7, 336.9, 337.6, 337.6, 336.9, 335.9, 334.5, 332.5, 331.0])
plt.plot(x1, T_K, "o", label="líquido"); plt.plot(y1, T_K, "s", label="vapor")
plt.xlabel("fracción de acetona"); plt.ylabel("T (K)"); plt.legend(); plt.show()

**Celda 7.8** · El coeficiente de actividad de cada punto

In [ ]:
t_C = T_K - 273.15; P_mmHg = 101.3 / 101.325 * 760
Psat1 = 10**(7.29958 - 1312.253/(t_C + 240.705))      # acetona, NIST (Ambrose y otros, 1974)
Psat2 = 10**(6.95465 - 1170.966/(t_C + 226.232))      # cloroformo, Felder y Rousseau
g1 = y1*P_mmHg / (x1*Psat1); g2 = (1 - y1)*P_mmHg / ((1 - x1)*Psat2)
print("gamma 1:", g1.round(3))
print("gamma 2:", g2.round(3))

**Celda 7.9** · least_squares con van Laar

In [ ]:
from scipy.optimize import least_squares
def van_laar(p, x):
    A12, A21 = p; z = A12*x + A21*(1 - x)
    return A12*(A21*(1 - x)/z)**2, A21*(A12*x/z)**2        # ln gamma 1, ln gamma 2
def residuos_vl(p):
    ln1, ln2 = van_laar(p, x1); return np.concatenate([np.log(g1) - ln1, np.log(g2) - ln2])
res_vl = least_squares(residuos_vl, [-0.5, -0.5])
print("A12, A21:", res_vl.x.round(3), "· estado:", res_vl.status, "· evaluaciones:", res_vl.nfev)

**Celda 7.10** · El modelo sobre los datos

In [ ]:
xx = np.linspace(0.01, 0.99, 99); ln1, ln2 = van_laar(res_vl.x, xx)
plt.plot(x1, np.log(g1), "o", label="ln gamma 1"); plt.plot(x1, np.log(g2), "s", label="ln gamma 2")
plt.plot(xx, ln1); plt.plot(xx, ln2)
plt.xlabel("fracción de acetona en el líquido"); plt.ylabel("ln gamma"); plt.legend(); plt.show()

**Celda 7.11** · Los residuos y la comparación con Perry

In [ ]:
print("suma de cuadrados del ajuste:", round(2*res_vl.cost, 5))     # cost es la mitad de la suma
print("con los de Perry (−0.8643, −0.5899):", round(np.sum(residuos_vl([-0.8643, -0.5899])**2), 5))
print("residuos de ln gamma 1:", res_vl.fun[:9].round(3))
print("residuos de ln gamma 2:", res_vl.fun[9:].round(3))

**Celda 7.12** · pcov: el error y la correlación de A, B y C

In [ ]:
errores = np.sqrt(np.diag(pcov))
correlacion = pcov / np.outer(errores, errores)
print("errores estándar de A, B, C:", errores.round(3))
print("correlación:"); print(correlacion.round(4))

**Celda 7.13** · Con la C fija, A y B quedan mejor determinados

In [ ]:
def antoine_C(T, A, B): return A - B/(T + C_nist)
p_AB, pcov_AB = curve_fit(antoine_C, T_dat, np.log10(P_dat), p0=[8, 1600])
print("A, B con la C de NIST:", p_AB.round(3))
print("errores estándar de A, B:", np.sqrt(np.diag(pcov_AB)).round(3))

**Celda 7.14** · La covarianza de least_squares

In [ ]:
J = res_vl.jac; n, k = J.shape                   # 18 residuos, 2 parámetros
s2 = 2*res_vl.cost / (n - k)                     # varianza residual
cov_vl = s2 * np.linalg.inv(J.T @ J)
err_vl = np.sqrt(np.diag(cov_vl))
print("errores estándar de A12, A21:", err_vl.round(3))
print("correlación:", round(cov_vl[0, 1] / (err_vl[0]*err_vl[1]), 3))

## Sección 8 · Optimizar un proceso: el modelo como función

**Celda 8.1** · Antoine de NIST en bar y kelvin

In [ ]:
def psat(T, A, B, C): return 10**(A - B/(T + C))     # bar, T en K (forma de NIST)
metanol_bajo = (5.20409, 1581.341, -33.50)             # 288 a 357 K, las de la Sección 7
metanol_alto = (5.15853, 1569.613, -34.846)            # 354 a 513 K
agua = (4.6543, 1435.264, -64.848)                     # 256 a 373 K
a65 = psat(338.15, *metanol_bajo) / psat(338.15, *agua)
a100 = psat(373.15, *metanol_alto) / psat(373.15, *agua)
print("alfa a 65 °C:", round(a65, 2), "· a 100 °C:", round(a100, 2), "· media geométrica:", round(np.sqrt(a65*a100), 2))

**Celda 8.2** · El punto de burbuja ideal

In [ ]:
from scipy.optimize import brentq
def burbuja_ideal(T, x1=0.5, P=1.01325):                 # P en bar
    return x1*psat(T, *metanol_bajo) + (1 - x1)*psat(T, *agua) - P
Tb_ideal = brentq(burbuja_ideal, 300, 380)
print("burbuja ideal:", round(Tb_ideal - 273.15, 1), "°C · experimental: 73.2 °C")

**Celda 8.3** · Los coeficientes de actividad con NRTL

In [ ]:
def gammas(x1, T):                                     # NRTL, parámetros de Aspen: tau = a + b/T
    t12, t21 = -0.693 + 172.987/T, 2.7322 - 617.269/T
    G12, G21 = np.exp(-0.3001*t12), np.exp(-0.3001*t21); x2 = 1 - x1
    gam1 = np.exp(x2**2*(t21*(G21/(x1 + x2*G21))**2 + t12*G12/(x2 + x1*G12)**2))
    gam2 = np.exp(x1**2*(t12*(G12/(x2 + x1*G12))**2 + t21*G21/(x1 + x2*G21)**2))
    return gam1, gam2

**Celda 8.4** · El punto de burbuja con NRTL

In [ ]:
def burbuja_nrtl(T, x1=0.5, P=1.01325):
    gam1, gam2 = gammas(x1, T)
    return x1*gam1*psat(T, *metanol_bajo) + (1 - x1)*gam2*psat(T, *agua) - P
Tb_nrtl = brentq(burbuja_nrtl, 300, 380)
print("burbuja con NRTL:", round(Tb_nrtl - 273.15, 1), "°C")
print("gammas:", np.round(gammas(0.5, Tb_nrtl), 3))

**Celda 8.5** · Fenske y Underwood

In [ ]:
alfa, xF, xD, xB, F = 3.68, 0.2365, 0.99, 0.01, 65.5      # F en kmol/h
N_min = np.log(xD/(1 - xD) * (1 - xB)/xB) / np.log(alfa)       # Fenske
R_min = (xD/xF - alfa*(1 - xD)/(1 - xF)) / (alfa - 1)         # Underwood, líquido saturado
print("N mín =", round(N_min, 2), "etapas · R mín =", round(R_min, 3))

**Celda 8.6** · Gilliland con la correlación de Molokanov

In [ ]:
def etapas(reflujo):
    X = (reflujo - R_min) / (reflujo + 1)
    Y = 1 - np.exp((1 + 54.4*X)/(11 + 117.2*X) * (X - 1)/np.sqrt(X))
    return (Y + N_min) / (1 - Y)
reflujo = 1.3*R_min; D = F*(xF - xB)/(xD - xB)
print("R =", round(reflujo, 3), "· N =", round(etapas(reflujo), 1), "etapas teóricas · D =", round(D, 1), "kmol/h")

**Celda 8.7** · Las etapas contra el reflujo

In [ ]:
razones = np.linspace(1.05, 3, 100)                 # R entre R mín
plt.plot(razones, etapas(razones*R_min))
plt.axvspan(1.2, 1.5, alpha=0.2)                    # la franja de la heurística
plt.xlabel("R / R mín"); plt.ylabel("etapas teóricas"); plt.show()
print("con R/R mín = 1.05, 1.2, 1.5, 2 y 3:", etapas(np.array([1.05, 1.2, 1.5, 2, 3])*R_min).round(1))

**Celda 8.8** · Un costo ilustrativo del reflujo

In [ ]:
def costo_columna(razon):                           # $/año, coeficientes ilustrativos
    capital = 5000*etapas(razon*R_min)              # por etapa teórica
    vapor = 3200*(razon*R_min + 1)*D                # por kmol/h de vapor en el rehervidor
    return capital + vapor
res_c = minimize_scalar(costo_columna, bounds=(1.02, 3), method="bounded")
print("R/R mín de menor costo:", round(res_c.x, 2), "· costo:", round(res_c.fun), "$/año")
print("con 1.3:", round(costo_columna(1.3)), "$/año")

**Celda 8.9** · Si cambia el precio del vapor

In [ ]:
for factor in (0.5, 1, 2):
    f = lambda razon: 5000*etapas(razon*R_min) + factor*3200*(razon*R_min + 1)*D
    r_opt = minimize_scalar(f, bounds=(1.02, 3), method="bounded").x
    print("vapor ×", factor, "→ R/R mín de menor costo:", round(r_opt, 2))
malla = np.linspace(1.02, 3, 2000); dentro = malla[costo_columna(malla) <= 1.01*res_c.fun]
print("a menos de 1 % del mínimo:", round(dentro.min(), 2), "a", round(dentro.max(), 2))

**Celda 8.10** · La planta con reciclo: el balance

In [ ]:
for X in (0.3, 0.5, 0.6, 0.8):                     # conversión por paso
    M = 100 / X; reciclo = 100*(1 - X)/X             # kmol/h, con 100 kmol/h frescos
    print("X =", X, "· al reactor:", round(M, 1), "· reciclo:", round(reciclo, 1), "kmol/h")

**Celda 8.11** · C(X): el reactor contra el reciclo

In [ ]:
a, b = 4, 9                                          # $/t, coeficientes del curso
def costo_X(v): X = v[0]; return a*X/(1 - X) + b*(1 - X)/X
conv = np.linspace(0.15, 0.9, 200)
plt.plot(conv, [costo_X([c]) for c in conv]); plt.xlabel("conversión por paso X"); plt.ylabel("$/t"); plt.show()
print("a mano: X* =", round(np.sqrt(b)/(np.sqrt(a) + np.sqrt(b)), 3), "· C* =", 2*np.sqrt(a*b), "$/t")

**Celda 8.12** · SLSQP con la restricción de selectividad

In [ ]:
res_libre = minimize(costo_X, [0.3], method="SLSQP", bounds=[(0.01, 0.99)])
print("sin restricción: X =", res_libre.x.round(4), "· C =", round(res_libre.fun, 3), "$/t")
selectividad = {"type": "ineq", "fun": lambda v: 0.5 - v[0]}     # X ≤ 0.5
res_X = minimize(costo_X, [0.3], method="SLSQP", bounds=[(0.01, 0.99)], constraints=[selectividad])
print("con X ≤ 0.5: X =", res_X.x.round(4), "· C =", round(res_X.fun, 3), "$/t")
print("multiplicador:", res_X.multipliers.round(3), "· a mano, |C'(0.5)|:", abs(a/0.5**2 - b/0.5**2))

## Sección 9 · Optimizar cuando los datos no son exactos

**Celda 9.1** · Cuánto DME producir: la utilidad de una semana

In [ ]:
precio, costo_var, rescate = 720, 420, 270      # USD/t, coeficientes ilustrativos
def utilidad(q, d):                             # USD por semana: producimos q, se piden d toneladas
    vendido = np.minimum(q, d)
    return precio*vendido + rescate*(q - vendido) - costo_var*q
print("producimos 100 t y se piden 100:", utilidad(100, 100), "USD")
print("producimos 100 t y se piden 60:", utilidad(100, 60), "USD")
print("producimos 60 t y se piden 100:", utilidad(60, 100), "USD")

**Celda 9.2** · La decisión determinista y una semana distinta

In [ ]:
q_malla = np.arange(0, 161)                     # toneladas
q_det = q_malla[np.argmax(utilidad(q_malla, 100))]
print("decisión determinista:", q_det, "t · utilidad prevista:", utilidad(q_det, 100), "USD")
for d in (60, 80, 100, 120, 140):
    print("  si se piden", d, "t →", utilidad(q_det, d), "USD")

**Celda 9.3** · La utilidad esperada con una demanda normal

In [ ]:
from scipy import stats
demanda = stats.norm(100, 20)                   # t por semana: media 100, desviación 20
def esperada(q): return demanda.expect(lambda d: utilidad(q, d))
print("utilidad esperada con q = 100:", round(esperada(100), 1), "USD")

**Celda 9.4** · La decisión estocástica: la razón crítica

In [ ]:
falta, sobra = precio - costo_var, costo_var - rescate   # USD por tonelada que faltó o que sobró
razon = falta / (falta + sobra)
q_sto = demanda.ppf(razon)                      # el cuantil de la razón crítica
res_sto = minimize_scalar(lambda q: -esperada(q), bounds=(60, 140), method="bounded")
print("razón crítica:", round(razon, 4), "· q* =", round(q_sto, 2), "t · con minimize_scalar:", round(res_sto.x, 2), "t")
print("utilidad esperada:", round(esperada(q_sto), 1), "USD · con q = 100:", round(esperada(100), 1), "USD")

**Celda 9.5** · La decisión robusta: el peor caso del intervalo

In [ ]:
peor = [min(utilidad(q, d) for d in np.linspace(60, 140, 81)) for q in q_malla]
q_rob = q_malla[np.argmax(peor)]
print("decisión robusta:", q_rob, "t · garantía:", max(peor), "USD por semana")
print("su utilidad esperada:", round(esperada(q_rob), 1), "USD")
print("precio de la robustez:", round(esperada(q_sto) - esperada(q_rob), 1), "USD por semana")

**Celda 9.6** · El peor caso y la esperada, lado a lado

In [ ]:
plt.plot(q_malla, peor, label="peor semana, d entre 60 y 140")
plt.plot(q_malla, [esperada(q) for q in q_malla], label="utilidad esperada, demanda normal")
plt.xlabel("toneladas producidas"); plt.ylabel("USD por semana"); plt.legend(); plt.show()

**Celda 9.7** · DRO: solo la media y la desviación (cota de Scarf)

In [ ]:
mu, sigma = 100, 20
def cota_scarf(q): t = q - mu; return (np.sqrt(sigma**2 + t**2) - t) / 2     # cota de lo que falta, en promedio
def garantia_dro(q): return (precio - rescate)*(mu - cota_scarf(q)) - sobra*q
res_dro = minimize_scalar(lambda q: -garantia_dro(q), bounds=(60, 140), method="bounded")
q_dro = mu + sigma/2*(np.sqrt(falta/sobra) - np.sqrt(sobra/falta))          # fórmula de Scarf
print("q* DRO:", round(res_dro.x, 2), "t · con la fórmula de Scarf:", round(q_dro, 2), "t")
print("garantía en promedio:", round(garantia_dro(q_dro), 1), "USD por semana")

**Celda 9.8** · La peor distribución: dos valores

In [ ]:
a, b = mu - sigma*np.sqrt(sobra/falta), mu + sigma*np.sqrt(falta/sobra)
pa = falta / (falta + sobra)
desv = np.sqrt(pa*(a - mu)**2 + (1 - pa)*(b - mu)**2)
print("D =", round(a, 2), "con prob.", round(pa, 3), "· D =", round(b, 2), "con prob.", round(1 - pa, 3))
print("media:", round(pa*a + (1 - pa)*b, 1), "· desviación:", round(desv, 1))
print("utilidad esperada de q DRO con ella:", round(pa*utilidad(q_dro, a) + (1 - pa)*utilidad(q_dro, b), 1), "USD")

**Celda 9.9** · Las cuatro decisiones con tres criterios

In [ ]:
decisiones = {"determinista": q_det, "estocástica": q_sto, "robusta": q_rob, "DRO": q_dro}
print("decisión · t · esperada (normal) · peor promedio (media y desv.) · peor semana (d = 60)")
for nombre, q in decisiones.items():
    print(nombre, "·", round(q, 1), "·", round(esperada(q)), "·", round(garantia_dro(q)), "·", round(utilidad(q, 60)))

**Celda 9.10** · Las cuatro decisiones sobre la curva esperada

In [ ]:
plt.plot(q_malla, [esperada(q) for q in q_malla], color="gray")
for nombre, q in decisiones.items():
    plt.plot(q, esperada(q), "o", label=nombre)
plt.xlabel("toneladas producidas"); plt.ylabel("utilidad esperada (USD)"); plt.legend(); plt.show()

## Sección 10 · Proyecto final

**Celda 10.1** · Los grados de libertad de la planta de cloruro de vinilo

In [ ]:
NC_cv = 3                                  # dicloroetano (EDC), cloruro de vinilo y cloruro de hidrógeno
incog_planta = 5*NC_cv + 1                 # flujo de cada componente en 5 corrientes, más el avance de la reacción
ecs_mezclador = NC_cv                      # un balance por componente
ecs_horno = NC_cv                          # sale lo que entra, más lo que forma la reacción
ecs_separador = NC_cv + 2*NC_cv            # balances, y 6 flujos que valen cero: el separador es perfecto
ecs_planta = ecs_mezclador + ecs_horno + ecs_separador
print("incógnitas:", incog_planta, "· ecuaciones:", ecs_planta, "· GL:", incog_planta - ecs_planta)

**Celda 10.2** · Fijamos la conversión por paso y el balance queda resuelto

In [ ]:
F_edc = 100.0                              # kmol/h de dicloroetano fresco
def balance_cv(X):                         # kmol/h: al horno, reciclo y cloruro de vinilo producido
    return F_edc/X, F_edc*(1 - X)/X, F_edc
print("GL al fijar la conversión por paso:", incog_planta - ecs_planta - 1)
for X_p in (0.40, 0.52, 0.70):
    al_horno, reciclo_p, producto = balance_cv(X_p)
    print("X =", X_p, "· al horno:", round(al_horno, 1), "· reciclo:", round(reciclo_p, 1), "· producto:", producto, "kmol/h")

**Celda 10.3** · Cuánto cloruro de vinilo produce la planta

In [ ]:
M_cv = 62.50                               # kg/kmol de cloruro de vinilo
t_hora = F_edc * M_cv / 1000               # t/h
t_semana, t_anio = t_hora*168, t_hora*8000 # semana continua de 168 h; 8000 h de operación al año
print("producción:", t_hora, "t/h ·", t_semana, "t por semana ·", t_anio, "t por año")

**Celda 10.4** · El costo: el horno contra el reciclo

In [ ]:
a_horno, b_reciclo = 24.0, 39.0            # USD/t de cloruro de vinilo, coeficientes ilustrativos
X_max = 0.52                               # cota de la conversión por paso: coque y subproductos
def costo_planta(X):                       # USD por tonelada de cloruro de vinilo
    return a_horno*X/(1 - X) + b_reciclo*(1 - X)/X
print("en la cota · horno:", round(a_horno*X_max/(1 - X_max), 2), "· reciclo:", round(b_reciclo*(1 - X_max)/X_max, 2),
      "· total:", round(costo_planta(X_max), 2), "USD/t")

**Celda 10.5** · Grafica antes de optimizar: la curva y la cota

In [ ]:
X_malla = np.linspace(0.3, 0.8, 501)
plt.plot(X_malla, costo_planta(X_malla))
plt.axvline(X_max, color="red", linestyle="--")            # la cota
plt.axvspan(X_max, 0.8, color="red", alpha=0.1)            # a la derecha de la cota no se puede operar
plt.xlabel("conversión por paso X"); plt.ylabel("USD por tonelada de cloruro de vinilo"); plt.show()
print("el más barato de la malla, sin mirar la cota:", round(X_malla[np.argmin(costo_planta(X_malla))], 3))

**Celda 10.6** · A mano: el óptimo libre queda fuera de la cota

In [ ]:
X_libre = np.sqrt(b_reciclo) / (np.sqrt(a_horno) + np.sqrt(b_reciclo))
C_libre = 2*np.sqrt(a_horno*b_reciclo)
print("óptimo libre: X* =", round(X_libre, 4), "· C* =", round(C_libre, 2), "USD/t")
print("¿respeta la cota?", X_libre <= X_max)

**Celda 10.7** · A mano: el multiplicador es la pendiente en la cota

In [ ]:
derivada = a_horno/(1 - X_max)**2 - b_reciclo/X_max**2     # C'(X) en la cota
print("en la cota: C =", round(costo_planta(X_max), 2), "USD/t · pendiente:", round(derivada, 2))
print("mu a mano = |C'(cota)| =", round(abs(derivada), 2), "USD/t por unidad de X")

**Celda 10.8** · SLSQP con la cota: el resultado completo

In [ ]:
limite = {"type": "ineq", "fun": lambda v: X_max - v[0]}      # X ≤ 0.52, escrita como g(X) ≥ 0
res_cv = minimize(lambda v: costo_planta(v[0]), [0.3], method="SLSQP",
                  bounds=[(0.01, 0.99)], constraints=[limite])
print(res_cv)

**Celda 10.9** · La auditoría: SLSQP contra la mano

In [ ]:
mu_cv = res_cv.multipliers[0]
print("X:  SLSQP", round(res_cv.x[0], 4), "· a mano", X_max)
print("C:  SLSQP", round(res_cv.fun, 4), "· a mano", round(costo_planta(X_max), 4))
print("mu: SLSQP", round(mu_cv, 4), "· a mano", round(abs(derivada), 4))

**Celda 10.10** · La curva con la cota y el multiplicador escritos sobre ella

In [ ]:
plt.plot(X_malla, costo_planta(X_malla)); plt.axvline(X_max, color="red", linestyle="--")
tangente = costo_planta(X_max) - mu_cv*(X_malla - X_max)    # recta con pendiente −mu en la cota
plt.plot(X_malla, tangente, ":", color="gray"); plt.ylim(55, 110)
plt.plot(X_max, res_cv.fun, "o"); plt.plot(X_libre, C_libre, "s")
plt.annotate(f"X = {X_max} · C = {res_cv.fun:.2f} USD/t · mu = {mu_cv:.2f}", (X_max, res_cv.fun), xytext=(0.33, 100))
plt.xlabel("conversión por paso X"); plt.ylabel("USD/t"); plt.show()

**Celda 10.11** · El precio sombra, comprobado: movemos la cota

In [ ]:
for paso in (0.01, 0.001):
    limite_mas = {"type": "ineq", "fun": lambda v: X_max + paso - v[0]}
    r = minimize(lambda v: costo_planta(v[0]), [0.3], method="SLSQP", bounds=[(0.01, 0.99)], constraints=[limite_mas])
    print("cota", round(X_max + paso, 3), "→ ahorro real:", round(res_cv.fun - r.fun, 4), "· estimación con mu:", round(mu_cv*paso, 4), "USD/t")

**Celda 10.12** · Lo que vale un punto de conversión al año

In [ ]:
ahorro_punto = costo_planta(X_max) - costo_planta(X_max + 0.01)    # USD/t, si la cota pasa de 0.52 a 0.53
print("un punto más de cota:", round(ahorro_punto, 3), "USD/t ·", round(ahorro_punto*t_anio), "USD al año")
print("lo que cuesta la cota frente al óptimo libre:", round(costo_planta(X_max) - C_libre, 3), "USD/t ·",
      round((costo_planta(X_max) - C_libre)*t_anio), "USD al año")

**Celda 10.13** · Una función que resuelve el problema con otros a y b

In [ ]:
def resolver(a_, b_):                      # óptimo con la cota y su multiplicador
    f = lambda v: a_*v[0]/(1 - v[0]) + b_*(1 - v[0])/v[0]
    r = minimize(f, [0.3], method="SLSQP", bounds=[(0.01, 0.99)], constraints=[limite])
    return r.x[0], r.fun, r.multipliers[0] + 0.0      # + 0.0 evita imprimir −0.0
print("caso base: X, C, mu =", np.round(resolver(a_horno, b_reciclo), 4))

**Celda 10.14** · a y b, veinte y treinta por ciento arriba y abajo

In [ ]:
casos = [(0.7, 1), (0.8, 1), (1.2, 1), (1.3, 1), (1, 0.7), (1, 0.8), (1, 1.2), (1, 1.3)]   # factores de a y de b
for fa, fb in casos:
    X_l = np.sqrt(fb*b_reciclo) / (np.sqrt(fa*a_horno) + np.sqrt(fb*b_reciclo))          # óptimo libre, a mano
    X_s, C_s, mu_s = resolver(fa*a_horno, fb*b_reciclo)
    print("a ×", fa, "· b ×", fb, "→ X libre:", round(X_l, 4), "· X:", round(X_s, 4), "· C:", round(C_s, 2), "· mu:", round(mu_s, 2))

**Celda 10.15** · Si el reciclo cuesta 30 % menos, la cota ya no toca

In [ ]:
for fb in (0.7, 1, 1.3):
    curva = a_horno*X_malla/(1 - X_malla) + fb*b_reciclo*(1 - X_malla)/X_malla
    plt.plot(X_malla, curva, label=f"b × {fb}")
    plt.plot(X_malla[np.argmin(curva)], curva.min(), "o", color="black")    # el mínimo libre de cada curva
plt.axvline(X_max, color="red", linestyle="--")
plt.xlabel("conversión por paso X"); plt.ylabel("USD/t"); plt.legend(); plt.show()

**Celda 10.16** · La semana de cloruro de vinilo: faltar contra sobrar

In [ ]:
precio_cv, costo_var_cv, rescate_cv = 850, 700, 450      # USD/t de cloruro de vinilo, ilustrativos
def utilidad_cv(q, d):                                   # USD por semana: producimos q, se piden d toneladas
    vendido = np.minimum(q, d)
    return precio_cv*vendido + rescate_cv*(q - vendido) - costo_var_cv*q
falta_cv, sobra_cv = precio_cv - costo_var_cv, costo_var_cv - rescate_cv
razon_cv = falta_cv / (falta_cv + sobra_cv)
print("faltar cuesta", falta_cv, "USD/t · sobrar cuesta", sobra_cv, "USD/t · razón crítica:", razon_cv)

**Celda 10.17** · La decisión determinista y la estocástica

In [ ]:
media_cv, desv_cv = 1000, 150                            # t por semana
demanda_cv = stats.norm(media_cv, desv_cv)
def esperada_cv(q): return demanda_cv.expect(lambda d: utilidad_cv(q, d))
q_det_cv, q_sto_cv = media_cv, demanda_cv.ppf(razon_cv)  # la media y el cuantil de la razón crítica
res_sto_cv = minimize_scalar(lambda q: -esperada_cv(q), bounds=(700, t_semana), method="bounded")
print("determinista:", q_det_cv, "t · esperada:", round(esperada_cv(q_det_cv)), "USD")
print("estocástica:", round(q_sto_cv, 1), "t · con minimize_scalar:", round(res_sto_cv.x, 1), "t · esperada:", round(esperada_cv(q_sto_cv)), "USD")

**Celda 10.18** · La decisión robusta: el peor caso del intervalo

In [ ]:
q_malla_cv = np.arange(0, t_semana + 1, 5)               # de 0 a la capacidad de la planta, cada 5 t
peor_cv = [min(utilidad_cv(q, d) for d in np.linspace(700, 1300, 61)) for q in q_malla_cv]
q_rob_cv = q_malla_cv[np.argmax(peor_cv)]
print("robusta:", round(q_rob_cv), "t · garantía:", round(max(peor_cv)), "USD por semana")
print("su esperada:", round(esperada_cv(q_rob_cv)), "USD · precio de la robustez:", round(esperada_cv(q_sto_cv) - esperada_cv(q_rob_cv)), "USD por semana")

**Celda 10.19** · DRO con la cota de Scarf, y su condición

In [ ]:
q_dro_cv = media_cv + desv_cv/2*(np.sqrt(falta_cv/sobra_cv) - np.sqrt(sobra_cv/falta_cv))     # fórmula de Scarf
def garantia_cv(q):                                      # utilidad en el peor promedio, con la media y la desviación
    s = q - media_cv; return (precio_cv - rescate_cv)*(media_cv - (np.sqrt(desv_cv**2 + s**2) - s)/2) - sobra_cv*q
res_dro_cv = minimize_scalar(lambda q: -garantia_cv(q), bounds=(700, t_semana), method="bounded")
punto_bajo = media_cv - desv_cv*np.sqrt(sobra_cv/falta_cv)   # el valor bajo de la peor distribución
print("DRO:", round(q_dro_cv, 1), "t · con minimize_scalar:", round(res_dro_cv.x, 1), "t · garantía en promedio:", round(garantia_cv(q_dro_cv)), "USD")
print("valor bajo de la peor distribución:", round(punto_bajo, 1), "t · ¿mayor o igual que cero?", punto_bajo >= 0)

**Celda 10.20** · Las cuatro decisiones con tres criterios

In [ ]:
decisiones_cv = {"determinista": q_det_cv, "estocástica": q_sto_cv, "robusta": q_rob_cv, "DRO": q_dro_cv}
print("decisión · t · esperada (normal) · peor promedio (media y desv.) · peor semana (d = 700)")
for nombre, q in decisiones_cv.items():
    print(nombre, "·", round(q, 1), "·", round(esperada_cv(q)), "·", round(garantia_cv(q)), "·", round(utilidad_cv(q, 700)))